# 01 — Dataset Exploration
Figures are saved to `reports/figures/01_dataset_exploration/` and shown inline.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, matplotlib.pyplot as plt, pandas as pd
from PIL import Image
from collections import Counter
import random

from src.datasets import _gather_samples, _train_val_test_split
from src.utils import fig_path, save_figure, save_results_csv, compute_metrics_per_class

%matplotlib inline

In [ ]:
DATA_ROOT    = '../datasets/raw'
SEED         = 42
SPLIT_RATIOS = (0.7, 0.15, 0.15)
REPORTS_DIR  = '../reports'
NB           = '01_dataset_exploration'  # subfolder name for figures

random.seed(SEED); np.random.seed(SEED)

## 1. Class balance

In [ ]:
paths, labels = _gather_samples(DATA_ROOT)
label_names   = {0: 'Human (Real)', 1: 'AI (Fake)'}
counts = Counter(labels); total = len(labels)
print(f'Total: {total}')
for lbl, name in label_names.items():
    n = counts[lbl]
    print(f'  {name}: {n} ({100*n/total:.1f}%)')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(list(label_names.values()), [counts[0], counts[1]], color=['steelblue', 'tomato'])
ax.set_title('Class Balance'); ax.set_ylabel('Images')
for i, lbl in enumerate([0, 1]):
    ax.text(i, counts[lbl] + total*0.005, str(counts[lbl]), ha='center')
fig.tight_layout()
save_figure(fig, fig_path(REPORTS_DIR, NB, 'class_balance.png'), show=True)

## 2. Split sizes + stratification check

In [ ]:
train_idx, val_idx, test_idx = _train_val_test_split(paths, labels, SPLIT_RATIOS, SEED)

split_records = []
for split_name, idx in [('Train', train_idx), ('Val', val_idx), ('Test', test_idx)]:
    sc = Counter([labels[i] for i in idx]); n = len(idx)
    split_records.append({'split': split_name, 'total': n,
        'n_human': sc[0], 'pct_human': round(100*sc[0]/n, 1),
        'n_ai':    sc[1], 'pct_ai':    round(100*sc[1]/n, 1)})
    print(f"{split_name:5s} {n:5d} | Human {sc[0]:4d} ({100*sc[0]/n:.1f}%) | AI {sc[1]:4d} ({100*sc[1]/n:.1f}%)")

ratios = [r['pct_human'] for r in split_records]
print(f'\nHuman% variance across splits: {np.var(ratios):.4f}  (ideal: 0.0)')

In [ ]:
df_splits = pd.DataFrame(split_records)
x = np.arange(len(df_splits)); w = 0.35
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - w/2, df_splits['pct_human'], w, label='Human (Real)', color='steelblue')
ax.bar(x + w/2, df_splits['pct_ai'],    w, label='AI (Fake)',    color='tomato')
ax.set_xticks(x); ax.set_xticklabels(df_splits['split'])
ax.set_ylabel('% of split'); ax.set_ylim(0, 100)
ax.set_title('Stratification check — class ratio per split')
ax.axhline(50, color='gray', linestyle='--', linewidth=0.8, label='50% ref')
ax.legend(); fig.tight_layout()
save_figure(fig, fig_path(REPORTS_DIR, NB, 'stratification_check.png'), show=True)

## 3. Image size distribution (sample 500)

In [ ]:
sample_paths = random.sample(paths, min(500, len(paths)))
widths, heights = [], []
for p in sample_paths:
    w, h = Image.open(p).size
    widths.append(w); heights.append(h)
print(f'Width  — min:{min(widths)} max:{max(widths)} mean:{np.mean(widths):.0f}')
print(f'Height — min:{min(heights)} max:{max(heights)} mean:{np.mean(heights):.0f}')
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(widths,  bins=40, color='steelblue', edgecolor='white'); axes[0].set_title('Width');  axes[0].set_xlabel('px')
axes[1].hist(heights, bins=40, color='salmon',    edgecolor='white'); axes[1].set_title('Height'); axes[1].set_xlabel('px')
fig.tight_layout()
save_figure(fig, fig_path(REPORTS_DIR, NB, 'size_distribution.png'), show=True)

## 4. Visual samples — Real vs AI

In [ ]:
def sample_by_class(paths, labels, label, n=4):
    idxs = [i for i, l in enumerate(labels) if l == label]
    return [paths[i] for i in random.sample(idxs, min(n, len(idxs)))]

n_cols = 4
fig, axes = plt.subplots(2, n_cols, figsize=(3*n_cols, 7))
for col, p in enumerate(sample_by_class(paths, labels, 0, n_cols)):
    axes[0, col].imshow(Image.open(p).convert('RGB')); axes[0, col].axis('off'); axes[0, col].set_title('Real', fontsize=9)
for col, p in enumerate(sample_by_class(paths, labels, 1, n_cols)):
    axes[1, col].imshow(Image.open(p).convert('RGB')); axes[1, col].axis('off'); axes[1, col].set_title('AI', fontsize=9)
fig.suptitle('Real vs AI-Generated', fontsize=12); fig.tight_layout()
save_figure(fig, fig_path(REPORTS_DIR, NB, 'samples_grid.png'), show=True)

## 5. Pixel intensity per class (200 images each)

In [ ]:
def mean_channel_hist(paths_list, resize=(64,64)):
    hists = {c: np.zeros(256) for c in ['R','G','B']}
    for p in paths_list:
        img = np.array(Image.open(p).convert('RGB').resize(resize))
        for ci, c in enumerate(['R','G','B']):
            h, _ = np.histogram(img[:,:,ci], bins=256, range=(0,256))
            hists[c] += h
    for c in hists: hists[c] /= hists[c].sum()
    return hists

real_hists = mean_channel_hist(sample_by_class(paths, labels, 0, 200))
ai_hists   = mean_channel_hist(sample_by_class(paths, labels, 1, 200))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
colors = {'R':'red','G':'green','B':'blue'}; x = np.arange(256)
for ax, ch in zip(axes, ['R','G','B']):
    ax.plot(x, real_hists[ch], color=colors[ch], label='Real', alpha=0.7)
    ax.plot(x, ai_hists[ch],   color=colors[ch], linestyle='--', label='AI', alpha=0.7)
    ax.set_title(f'{ch} channel'); ax.set_xlabel('Pixel value'); ax.legend()
fig.suptitle('Pixel intensity — Real vs AI', fontsize=12); fig.tight_layout()
save_figure(fig, fig_path(REPORTS_DIR, NB, 'intensity_distribution.png'), show=True)

## 6. Save stats to CSV

In [ ]:
save_results_csv(split_records, f'{REPORTS_DIR}/01_dataset_stats.csv')
print('Done. Figures saved to reports/figures/01_dataset_exploration/')